# Vector Search Inference Benchmark — SDK SP

Benchmarks and runs batch Vector Search inference using `VectorSearchClient` with the `shm-sp` service principal.

**Two modes:**

| Cell | Mode | When to use |
|---|---|---|
| Cell 8 | Single-process sliding window | Benchmarking / concurrency tuning |
| Cell 9 | 4-worker parallel fan-out | Production (7.7 M rows) |

| Feature | Detail |
|---|---|
| Client | `VectorSearchClient` SDK with SP OAuth (`shm/sp-id` + `shm/sp-key`) |
| Concurrency | Fixed sliding window — `initial_concurrency=500` for single-process, `125` per worker for fan-out |
| Throughput | ~260 QPS single-process; ~528 QPS inference-window with 4-worker fan-out |
| Endpoint | `vs` at 3,000 QPS; per-client sweet spot ≈ 500 simultaneous connections |
| Output | Delta table with `method`, `auth_mode`, latency, matched IDs, and scores |

In [0]:
%pip install --quiet aiohttp databricks-ai-search databricks-sdk
%restart_python

In [0]:
# Defines widgets with sensible defaults for interactive use.
# Jobs can override any of these values.
dbutils.widgets.text("index_name",          "shm_skunkworks_catalog.vector_search.index",              "VS index (catalog.schema.index)")
dbutils.widgets.text("endpoint_name",       "vs",                                                      "VS endpoint name")
dbutils.widgets.text("output_table",        "shm_skunkworks_catalog.vector_search.combined_benchmark", "Output Delta table (catalog.schema.table)")
dbutils.widgets.text("id_column",           "registeredAccountId",                                     "ID column in source table")
dbutils.widgets.text("vector_column",       "query_vec",                                               "Vector column in source table")
dbutils.widgets.text("initial_concurrency", "500",                                                     "Starting concurrency")
dbutils.widgets.text("top_k",               "10",                                                      "Number of neighbors to return")
dbutils.widgets.text("batch_size",          "2000",                                                    "Rows per async batch")
dbutils.widgets.text("inference_month",     "2025-01",                                                 "Inference month label")
dbutils.widgets.text("max_rows",            "50000",                                                   "Max rows to benchmark (0 = all rows)")

In [0]:
import asyncio
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone

import aiohttp
import pandas as pd
from databricks.ai_search.client import VectorSearchClient
from databricks.sdk import WorkspaceClient
from pyspark.sql.types import ArrayType, DoubleType, StringType, StructField, StructType, TimestampType

# ── Widgets ─────────────────────────────────────────────────────────────
INDEX_NAME          = dbutils.widgets.get("index_name")
ENDPOINT_NAME       = dbutils.widgets.get("endpoint_name")
OUTPUT_TABLE        = dbutils.widgets.get("output_table")
ID_COLUMN           = dbutils.widgets.get("id_column")
VECTOR_COLUMN       = dbutils.widgets.get("vector_column")
INITIAL_CONCURRENCY = int(dbutils.widgets.get("initial_concurrency"))
TOP_K               = int(dbutils.widgets.get("top_k"))
BATCH_SIZE          = int(dbutils.widgets.get("batch_size"))
INFERENCE_MONTH     = dbutils.widgets.get("inference_month")
MAX_ROWS            = int(dbutils.widgets.get("max_rows"))
SOURCE_TABLE        = "shm_skunkworks_catalog.vector_search.prevectorized_table"

assert OUTPUT_TABLE,  "output_table widget must be set before running"
assert ENDPOINT_NAME, "endpoint_name widget must be set before running"

# ── Tuning constants ─────────────────────────────────────────────────────
MIN_CONCURRENCY    = 5
MAX_RETRIES        = 5
BACKOFF_THRESHOLD  = 0.05
SCALE_UP_THRESHOLD = 0.01

# ── Auth ─────────────────────────────────────────────────────────────────
w          = WorkspaceClient()
HOST       = w.config.host.rstrip("/")
TOKEN      = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
API_URL    = f"{HOST}/api/2.0/vector-search/indexes/{INDEX_NAME}/query"
API_HEADERS = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

SP_CLIENT_ID     = dbutils.secrets.get(scope="shm", key="sp-id")
SP_CLIENT_SECRET = dbutils.secrets.get(scope="shm", key="sp-key")

# ── SP OAuth token (bearer, no PAT rate-limit) ──────────────────────────
import requests as _req

def _fetch_sp_token(host, client_id, client_secret):
    """Exchange SP client credentials for a short-lived Databricks OAuth token."""
    r = _req.post(
        f"{host}/oidc/v1/token",
        data={
            "grant_type":    "client_credentials",
            "client_id":     client_id,
            "client_secret": client_secret,
            "scope":         "all-apis",
        },
        timeout=30,
    )
    r.raise_for_status()
    return r.json()["access_token"]

SP_TOKEN       = _fetch_sp_token(HOST, SP_CLIENT_ID, SP_CLIENT_SECRET)
SP_API_HEADERS = {"Authorization": f"Bearer {SP_TOKEN}", "Content-Type": "application/json"}

print(f"Host:           {HOST}")
print(f"Endpoint:       {ENDPOINT_NAME}")
print(f"Index:          {INDEX_NAME}")
print(f"Source table:   {SOURCE_TABLE}")
print(f"Output table:   {OUTPUT_TABLE}")
print(f"SP client_id:   {'[loaded]' if SP_CLIENT_ID else '[MISSING — check shm/sp-id]'}")
print(f"batch_size={BATCH_SIZE:,}  top_k={TOP_K}  initial_concurrency={INITIAL_CONCURRENCY}")

In [0]:
OUTPUT_SCHEMA = StructType([
    StructField(ID_COLUMN,             StringType(),            False),
    StructField("inference_month",     StringType(),            False),
    StructField("method",              StringType(),            False),
    StructField("auth_mode",           StringType(),            False),
    StructField("request_latency_ms",  DoubleType(),            True),
    StructField("matched_account_ids", ArrayType(StringType()), True),
    StructField("search_scores",       ArrayType(DoubleType()), True),
    StructField("processed_at",        TimestampType(),         True),
])


def ensure_output_table():
    if not spark.catalog.tableExists(OUTPUT_TABLE):
        spark.createDataFrame([], OUTPUT_SCHEMA).write.format("delta").mode("error").saveAsTable(OUTPUT_TABLE)
        print(f"Created output table: {OUTPUT_TABLE}")


def load_source_rows():
    rows_df = spark.table(SOURCE_TABLE).select(ID_COLUMN, VECTOR_COLUMN).toPandas()
    rows_df[ID_COLUMN] = rows_df[ID_COLUMN].astype(str)
    if MAX_ROWS > 0:
        rows_df = rows_df.iloc[:MAX_ROWS].reset_index(drop=True)
    return rows_df


def write_results(out_rows):
    if out_rows:
        (
            spark
            .createDataFrame(pd.DataFrame(out_rows), schema=OUTPUT_SCHEMA)
            .write.format("delta").mode("append")
            .saveAsTable(OUTPUT_TABLE)
        )


print("Output schema ready.")
print(f"  combined output → {OUTPUT_TABLE}")

In [0]:
def extract_hits(result):
    manifest_cols = [c["name"] for c in result.get("manifest", {}).get("columns", [])]
    data_rows     = result.get("result", {}).get("data_array", [])

    id_idx    = manifest_cols.index(ID_COLUMN) if ID_COLUMN in manifest_cols else 0
    score_idx = manifest_cols.index("score")   if "score"   in manifest_cols else -1

    return [
        {"neighbor_id": str(r[id_idx]), "search_score": float(r[score_idx])}
        for r in data_rows
    ]


class AsyncTokenBucket:
    """Queue-based async rate limiter — no thundering herd.

    A single background refill task wakes on a fixed interval and drops
    tokens into a bounded ``asyncio.Queue``.  Workers call
    ``await bucket.acquire()`` which is a plain ``queue.get()``; exactly
    ONE waiter is unblocked per token — no lock polling, no N-waiter
    wake-storms when many coroutines are queued.

    Usage::

        bucket = AsyncTokenBucket(rate=2800.0, burst=50)
        bucket.start()          # launch background refill task
        await bucket.acquire()  # gate each outbound request
        bucket.stop()           # cancel refill task when done

    Parameters
    ----------
    rate  : tokens / second (sustained throughput target)
    burst : pre-filled tokens at startup — keep small (≤100) to avoid
            an initial flood that overwhelms the endpoint
    """
    def __init__(self, rate: float, burst: int = 50):
        self._rate     = float(rate)
        self._burst    = int(burst)
        # Produce tokens in batches so asyncio.sleep fires at ~10 ms intervals
        # rather than once per token (too fast at 2800/s).
        self._batch    = max(1, int(self._rate / 100))   # e.g. 28 at rate=2800
        self._interval = self._batch / self._rate         # ≈ 10 ms
        maxsize        = max(self._burst, self._batch * 2)
        self._queue    = asyncio.Queue(maxsize=maxsize)
        self._task     = None

    def start(self) -> None:
        """Pre-fill burst tokens and launch the background refill task."""
        for _ in range(self._burst):
            try:
                self._queue.put_nowait(1)
            except asyncio.QueueFull:
                break
        self._task = asyncio.create_task(self._refill())

    async def _refill(self) -> None:
        while True:
            await asyncio.sleep(self._interval)
            for _ in range(self._batch):
                try:
                    self._queue.put_nowait(1)
                except asyncio.QueueFull:
                    break   # bucket full — drop excess tokens

    async def acquire(self) -> None:
        """Block until a token is available. O(1), wakes exactly one waiter."""
        await self._queue.get()

    def stop(self) -> None:
        """Cancel the background refill task."""
        if self._task and not self._task.done():
            self._task.cancel()


async def query_one_aiohttp(session, sem, row_id, vec, counters, req_headers=None):
    payload = {
        "query_vector": vec.tolist() if hasattr(vec, "tolist") else list(vec),
        "columns": [ID_COLUMN],
        "num_results": TOP_K,
    }

    async with sem:
        for attempt in range(MAX_RETRIES):
            started = time.perf_counter()
            try:
                async with session.post(API_URL, json=payload, headers=req_headers or API_HEADERS) as resp:
                    latency_ms = (time.perf_counter() - started) * 1000

                    if resp.status in (429, 503):
                        counters["rate_limited"] += 1
                        await asyncio.sleep(min(0.5 * (2 ** attempt), 30.0))
                        continue

                    if resp.status != 200:
                        counters["errors"] += 1
                        return row_id, [], 0.0

                    return row_id, extract_hits(await resp.json()), latency_ms

            except (aiohttp.ClientError, asyncio.TimeoutError):
                counters["errors"] += 1
                if attempt == MAX_RETRIES - 1:
                    return row_id, [], 0.0
                await asyncio.sleep(0.5 * (2 ** attempt))

    return row_id, [], 0.0


def sdk_similarity_search_timed(index, query_vec):
    started = time.perf_counter()
    result = index.similarity_search(
        query_vector=query_vec,
        columns=[ID_COLUMN],
        num_results=TOP_K,
    )
    return result, (time.perf_counter() - started) * 1000


async def query_one_sdk(loop, executor, sem, index, row_id, vec, counters):
    query_vec = vec.tolist() if hasattr(vec, "tolist") else list(vec)

    async with sem:
        for attempt in range(MAX_RETRIES):
            try:
                result, latency_ms = await loop.run_in_executor(
                    executor,
                    lambda v=query_vec: sdk_similarity_search_timed(index, v),
                )
                return row_id, extract_hits(result), latency_ms

            except Exception as exc:
                s = str(exc).lower()
                is_rate = any(t in s for t in ("429", "503", "too many requests", "rate limit", "throttl"))
                if is_rate:
                    counters["rate_limited"] += 1
                    await asyncio.sleep(min(0.5 * (2 ** attempt), 30.0))
                    continue
                counters["errors"] += 1
                if attempt == MAX_RETRIES - 1:
                    return row_id, [], 0.0
                await asyncio.sleep(0.5 * (2 ** attempt))

    return row_id, [], 0.0

In [0]:
def adjust_concurrency(current_concurrency, counters, batch_size):
    rate = counters["rate_limited"] / max(batch_size, 1)
    if rate > BACKOFF_THRESHOLD:
        return max(MIN_CONCURRENCY, current_concurrency // 2)
    if rate < SCALE_UP_THRESHOLD:
        return int(current_concurrency * 1.25)
    return current_concurrency


async def run_profile(label, method, auth_mode, rows_df, task_builder):
    current_concurrency = INITIAL_CONCURRENCY
    total_rows          = len(rows_df)
    processed           = 0
    total_latency_ms    = 0.0
    total_requests      = 0
    rl_total            = 0
    err_total           = 0
    job_started         = time.perf_counter()

    print(f"\n{'='*72}")
    print(f"  Profile run: {label}")
    print(f"{'='*72}")

    for batch_id, start in enumerate(range(0, total_rows, BATCH_SIZE)):
        batch    = rows_df.iloc[start : start + BATCH_SIZE]
        counters = {"rate_limited": 0, "errors": 0}
        sem      = asyncio.Semaphore(current_concurrency)
        results  = await task_builder(batch, sem, counters)

        batch_lats       = [lat for _, _, lat in results if lat > 0]
        total_latency_ms += sum(batch_lats)
        total_requests   += len(batch_lats)
        rl_total         += counters["rate_limited"]
        err_total        += counters["errors"]

        processed_at = datetime.now(timezone.utc)
        write_results([
            {
                ID_COLUMN:             row_id,
                "inference_month":     INFERENCE_MONTH,
                "method":              method,
                "auth_mode":           auth_mode,
                "request_latency_ms":  float(latency_ms) if latency_ms > 0 else None,
                "matched_account_ids": [h["neighbor_id"] for h in hits],
                "search_scores":       [h["search_score"] for h in hits],
                "processed_at":        processed_at,
            }
            for row_id, hits, latency_ms in results
        ])

        processed += len(batch)
        elapsed_s = max(time.perf_counter() - job_started, 1e-9)
        avg_lat   = (sum(batch_lats) / len(batch_lats)) if batch_lats else 0.0
        qps       = processed / elapsed_s
        pct       = processed / total_rows * 100

        print(
            f"  [batch {batch_id + 1:>3}] {processed:>6,}/{total_rows:,} ({pct:5.1f}%) | "
            f"avg={avg_lat:7.1f}ms | qps={qps:7.1f} | conc={current_concurrency:>4} | "
            f"rl={counters['rate_limited']:>4} | err={counters['errors']:>3}"
        )
        current_concurrency = adjust_concurrency(current_concurrency, counters, len(batch))

    elapsed_s   = time.perf_counter() - job_started
    overall_avg = total_latency_ms / total_requests if total_requests else 0.0
    overall_qps = total_rows / elapsed_s

    print(f"\n  >> {total_rows:,} rows | {elapsed_s:.1f}s | avg={overall_avg:.1f}ms | qps={overall_qps:.1f}")
    print(f"  >> rate_limited={rl_total}  errors={err_total}  output={OUTPUT_TABLE}")

    return {
        "label":            label,
        "method":           method,
        "auth_mode":        auth_mode,
        "rows":             total_rows,
        "elapsed_s":        round(elapsed_s, 2),
        "avg_latency_ms":   round(overall_avg, 2),
        "qps":              round(overall_qps, 2),
        "rate_limited":     rl_total,
        "errors":           err_total,
        "output_table":     OUTPUT_TABLE,
    }


async def run_profile_sw(label, method, auth_mode, rows_df, make_task_fn):
    """Sliding-window inference pass.

    Keeps exactly INITIAL_CONCURRENCY requests in flight at all times by
    scheduling all row tasks up-front and draining via asyncio.as_completed.
    A completion immediately unblocks the next waiting task — no batch
    boundary stalls, no straggler penalty.

    make_task_fn(row_id, vec, counters, sem) must return a coroutine that
    resolves to (row_id, hits, latency_ms).

    Concurrency is fixed for the duration of the run (no adaptive scaling).
    Rate-limit events are reported at each checkpoint so you can see the
    threshold without self-throttling.
    """
    total_rows = len(rows_df)
    sem        = asyncio.Semaphore(INITIAL_CONCURRENCY)
    counters   = {"rate_limited": 0, "errors": 0}
    rl_total   = 0
    err_total  = 0

    # Schedule every row immediately; semaphore caps in-flight count.
    tasks = [
        asyncio.create_task(make_task_fn(row_id, vec, counters, sem))
        for row_id, vec in zip(rows_df[ID_COLUMN].tolist(), rows_df[VECTOR_COLUMN].tolist())
    ]

    completed      = 0
    total_lat      = 0.0
    total_reqs     = 0
    results_buffer = []
    checkpoint     = 0
    job_started    = time.perf_counter()

    print(f"\n{'='*72}")
    print(f"  Profile: {label}  [sliding window | conc={INITIAL_CONCURRENCY}]")
    print(f"{'='*72}")

    for fut in asyncio.as_completed(tasks):
        row_id, hits, latency_ms = await fut
        completed += 1
        if latency_ms > 0:
            total_lat  += latency_ms
            total_reqs += 1

        results_buffer.append({
            ID_COLUMN:             row_id,
            "inference_month":     INFERENCE_MONTH,
            "method":              method,
            "auth_mode":           auth_mode,
            "request_latency_ms":  float(latency_ms) if latency_ms > 0 else None,
            "matched_account_ids": [h["neighbor_id"] for h in hits],
            "search_scores":       [h["search_score"] for h in hits],
            "processed_at":        datetime.now(timezone.utc),
        })

        if len(results_buffer) >= BATCH_SIZE or completed == total_rows:
            write_results(results_buffer)
            results_buffer.clear()
            checkpoint  += 1
            elapsed_s    = max(time.perf_counter() - job_started, 1e-9)
            avg_lat      = total_lat / total_reqs if total_reqs else 0.0
            qps          = completed / elapsed_s
            rl_ckpt      = counters["rate_limited"]
            err_ckpt     = counters["errors"]
            rl_total    += rl_ckpt
            err_total   += err_ckpt
            counters["rate_limited"] = 0
            counters["errors"]       = 0
            print(
                f"  [ckpt {checkpoint:>3}] {completed:>6,}/{total_rows:,}"
                f" ({completed/total_rows*100:5.1f}%) | "
                f"avg={avg_lat:7.1f}ms | qps={qps:7.1f} | conc={INITIAL_CONCURRENCY:>5} | "
                f"rl={rl_ckpt:>4} | err={err_ckpt:>3}"
            )

    elapsed_s   = time.perf_counter() - job_started
    overall_avg = total_lat / total_reqs if total_reqs else 0.0
    overall_qps = total_rows / elapsed_s
    print(f"\n  >> {total_rows:,} rows | {elapsed_s:.1f}s | avg={overall_avg:.1f}ms | qps={overall_qps:.1f}")
    print(f"  >> rate_limited={rl_total}  errors={err_total}  output={OUTPUT_TABLE}")

    return {
        "label":          label,
        "method":         method,
        "auth_mode":      auth_mode,
        "rows":           total_rows,
        "elapsed_s":      round(elapsed_s, 2),
        "avg_latency_ms": round(overall_avg, 2),
        "qps":            round(overall_qps, 2),
        "rate_limited":   rl_total,
        "errors":         err_total,
        "output_table":   OUTPUT_TABLE,
    }

In [0]:
ensure_output_table()
all_rows = load_source_rows()
print(f"Rows to benchmark: {len(all_rows):,}  (max_rows={MAX_ROWS}, 0=all)")

comparison = []

loop = asyncio.get_event_loop()

import builtins as _builtins
_orig_print = _builtins.print
_builtins.print = lambda *a, **kw: (
    None if "[NOTICE]" in " ".join(str(x) for x in a)
    else _orig_print(*a, **kw)
)
try:
    vsc_sp = VectorSearchClient(
        workspace_url=HOST,
        service_principal_client_id=SP_CLIENT_ID,
        service_principal_client_secret=SP_CLIENT_SECRET,
    )

    for label, auth_mode, vsc in [
        ("sdk-sp", "shm-sp", vsc_sp),
    ]:
        index = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
        # Thread pool sized exactly to concurrency — no over-allocation needed
        # since the sliding window never exceeds INITIAL_CONCURRENCY in-flight.
        with ThreadPoolExecutor(max_workers=max(INITIAL_CONCURRENCY, 64)) as executor:
            async def sdk_single(row_id, vec, counters, sem, _index=index, _executor=executor):
                return await query_one_sdk(loop, _executor, sem, _index, row_id, vec, counters)

            comparison.append(
                await run_profile_sw(label, "sdk", auth_mode, all_rows.copy(), sdk_single)
            )
finally:
    _builtins.print = _orig_print

print(f"\n{'='*88}")
print("  PROFILING COMPARISON")
print(f"{'='*88}")
print(f"  {'profile':<18} {'elapsed_s':>10} {'avg_request_ms':>16} {'qps':>10} {'rate_limited':>14} {'errors':>8}")
print(f"  {'-'*82}")
for m in comparison:
    print(
        f"  {m['label']:<18} {m['elapsed_s']:>10} {m['avg_latency_ms']:>16} "
        f"{m['qps']:>10} {m['rate_limited']:>14} {m['errors']:>8}"
    )
print(f"{'='*88}")

In [0]:
# Fan out N independent notebook workers via dbutils.notebook.run().
# Each worker runs in its own serverless session (separate Python process + GIL),
# processes 1/N of the rows, and writes directly to the output table.
# Aggregate QPS ≈ N × single-worker QPS (currently ~260 at conc=500).

import json
import time
from concurrent.futures import ThreadPoolExecutor as _TPE

N_WORKERS        = 4    # validated sweet spot: 4 × conc=125 = 500 total connections → ~528 QPS inference-window
WORKER_CONC      = 125  # per-worker concurrency — independent of the single-process INITIAL_CONCURRENCY widget
WORKER_PATH      = "/Users/scott.mckean@databricks.com/3c_worker"

def _run_worker(partition_id):
    result_json = dbutils.notebook.run(
        WORKER_PATH,
        timeout_seconds=7200,
        arguments={
            "partition_id":        str(partition_id),
            "n_partitions":        str(N_WORKERS),
            "host":                HOST,
            "endpoint_name":       ENDPOINT_NAME,
            "index_name":          INDEX_NAME,
            "source_table":        SOURCE_TABLE,
            "output_table":        OUTPUT_TABLE,
            "id_column":           ID_COLUMN,
            "vector_column":       VECTOR_COLUMN,
            "max_rows":            str(MAX_ROWS),
            "initial_concurrency": str(WORKER_CONC),
            "top_k":               str(TOP_K),
            "inference_month":     INFERENCE_MONTH,
        },
    )
    return json.loads(result_json)

print(f"Launching {N_WORKERS} parallel notebook workers")
print(f"  conc={WORKER_CONC}/worker | max_rows={MAX_ROWS} | expected inference QPS ≈ {N_WORKERS * 127:,}")
print()
print()

_t0 = time.perf_counter()
with _TPE(max_workers=N_WORKERS) as _pool:
    _futures = [_pool.submit(_run_worker, i) for i in range(N_WORKERS)]
    _results = []
for _f in _futures:
    try:
        _results.append(_f.result())
    except Exception as _e:
        _results.append({"partition_id": -1, "rows": 0, "elapsed_s": 0,
                         "qps": 0, "rate_limited": 0, "errors": -1,
                         "_error": str(_e)[:200]})
_wall = time.perf_counter() - _t0

_total_rows = sum(r["rows"] for r in _results)
_agg_qps    = _total_rows / _wall

print(f"\n{'='*72}")
print(f"  Fan-out complete")
print(f"  {N_WORKERS} workers | {_total_rows:,} rows | wall={_wall:.1f}s | aggregate QPS={_agg_qps:.1f}")
print(f"{'='*72}")
for r in sorted(_results, key=lambda x: x["partition_id"]):
    print(
        f"  worker {r['partition_id'] + 1}: {r['rows']:>6,} rows | "
        f"{r['elapsed_s']:6.1f}s | {r['qps']:6.1f} QPS | "
        f"rl={r['rate_limited']} | err={r['errors']}"
    )

In [0]:
# Each Spark partition runs in a separate Python process — no shared GIL.
# NUM_PARTITIONS × ~170 QPS (per-partition sweet spot) approaches the 1,000 QPS ceiling.

import time as _time

NUM_PARTITIONS     = 8    # one per worker; measured ~115 QPS total on serverless (single executor); prefer cell 9 fan-out for production
PER_PARTITION_CONC = 500  # matches single-process sweet spot

# Capture driver-side values into the closure (serialised to each worker).
_host    = HOST
_ep      = ENDPOINT_NAME
_idx     = INDEX_NAME
_sp_id   = SP_CLIENT_ID
_sp_sec  = SP_CLIENT_SECRET
_id_col  = ID_COLUMN
_vec_col = VECTOR_COLUMN
_top_k   = TOP_K
_month   = INFERENCE_MONTH
_conc    = PER_PARTITION_CONC


def _run_partition(partition_iter):
    """Runs inside a Spark executor (separate Python process per task)."""
    import asyncio
    import builtins
    import time
    from concurrent.futures import ThreadPoolExecutor
    from datetime import datetime, timezone

    import pandas as pd
    from databricks.ai_search.client import VectorSearchClient

    # Suppress SDK notices in the worker
    _orig = builtins.print
    builtins.print = lambda *a, **kw: (
        None if "[NOTICE]" in " ".join(str(x) for x in a) else _orig(*a, **kw)
    )
    try:
        rows = pd.concat(list(partition_iter), ignore_index=True)
    finally:
        builtins.print = _orig

    _EMPTY_COLS = [_id_col, "inference_month", "method", "auth_mode",
                   "request_latency_ms", "matched_account_ids", "search_scores", "processed_at"]
    if rows.empty:
        yield pd.DataFrame(columns=_EMPTY_COLS)
        return

    # Fresh VS client per worker
    vsc = VectorSearchClient(
        workspace_url=_host,
        service_principal_client_id=_sp_id,
        service_principal_client_secret=_sp_sec,
    )
    index        = vsc.get_index(endpoint_name=_ep, index_name=_idx)
    part_started = time.perf_counter()

    def _sdk_timed(qvec):
        t0  = time.perf_counter()
        res = index.similarity_search(query_vector=qvec, columns=[_id_col], num_results=_top_k)
        return res, (time.perf_counter() - t0) * 1000

    def _hits(res):
        cols  = [c["name"] for c in res.get("manifest", {}).get("columns", [])]
        drows = res.get("result", {}).get("data_array", [])
        ii    = cols.index(_id_col) if _id_col in cols else 0
        si    = cols.index("score") if "score" in cols else -1
        return [{"neighbor_id": str(r[ii]), "search_score": float(r[si])} for r in drows]

    async def _one(row_id, vec, sem, loop, ex):
        qvec = vec.tolist() if hasattr(vec, "tolist") else list(vec)
        async with sem:
            for attempt in range(5):
                try:
                    res, lat = await loop.run_in_executor(ex, lambda v=qvec: _sdk_timed(v))
                    return row_id, _hits(res), lat
                except Exception as exc:
                    s = str(exc).lower()
                    if any(t in s for t in ("429", "503", "rate limit", "throttl")):
                        await asyncio.sleep(min(0.5 * (2 ** attempt), 30.0))
                        continue
                    if attempt == 4:
                        return row_id, [], 0.0
                    await asyncio.sleep(0.5 * (2 ** attempt))
        return row_id, [], 0.0

    async def _run_all():
        loop = asyncio.get_event_loop()
        sem  = asyncio.Semaphore(_conc)
        ex   = ThreadPoolExecutor(max_workers=_conc)
        tasks = [
            asyncio.create_task(_one(str(row_id), vec, sem, loop, ex))
            for row_id, vec in zip(rows[_id_col].tolist(), rows[_vec_col].tolist())
        ]
        out = [await fut for fut in asyncio.as_completed(tasks)]
        ex.shutdown(wait=False)
        return out

    results = asyncio.run(_run_all())

    elapsed = time.perf_counter() - part_started
    n       = len(results)
    _orig(f"[partition] {n:,} rows | {elapsed:.1f}s | {n/elapsed:.1f} QPS")

    ts     = datetime.now(timezone.utc)
    out_df = pd.DataFrame([
        {
            _id_col:               row_id,
            "inference_month":     _month,
            "method":              "sdk-sp-spark",
            "auth_mode":           "shm-sp",
            "request_latency_ms":  float(lat) if lat > 0 else None,
            "matched_account_ids": [h["neighbor_id"] for h in hits],
            "search_scores":       [h["search_score"] for h in hits],
            "processed_at":        ts,
        }
        for row_id, hits, lat in results
    ])
    yield out_df


# ── Run ───────────────────────────────────────────────────────────────────
spark_started = _time.perf_counter()

src = spark.table(SOURCE_TABLE).select(ID_COLUMN, VECTOR_COLUMN)
if MAX_ROWS > 0:
    src = src.limit(MAX_ROWS)

print(f"Launching {NUM_PARTITIONS} partitions | conc={PER_PARTITION_CONC} per partition")
print(f"Expected QPS ≈ {NUM_PARTITIONS * 115:,}  (endpoint ceiling is 3,000 QPS; Spark approach limited by executor count on serverless)")
print()

(
    src.repartition(NUM_PARTITIONS)
       .mapInPandas(_run_partition, schema=OUTPUT_SCHEMA)
       .write.mode("append")
       .saveAsTable(OUTPUT_TABLE)
)

spark_elapsed = _time.perf_counter() - spark_started
print(f"\n[spark-sp] {MAX_ROWS:,} rows | {NUM_PARTITIONS} partitions | conc={PER_PARTITION_CONC}/partition")
print(f"  elapsed={spark_elapsed:.1f}s | effective QPS={MAX_ROWS/spark_elapsed:.1f}")

## Benchmark design notes

### What cell 8 measures

**sdk-sp** — `VectorSearchClient` with the `shm-sp` service principal, sliding-window concurrency.

### Latency definition

`avg_request_ms` is **cumulative** across all completions so far (not per-checkpoint).
Timing is captured inside the worker thread around `similarity_search(...)`, excluding thread-pool queue delay.

Because `asyncio.as_completed` returns the fastest tasks first, the cumulative average grows over time as slower tasks drain last — this is expected, not a sign of degradation.

### Concurrency sweet spot (3,000 QPS endpoint)

| `initial_concurrency` | Outcome |
|---|---|
| 300 | 206 QPS, stable |
| **500** | **260 QPS, stable — recommended** |
| 750 | Endpoint congestion cliff at ckpt 4 |
| 2,000+ | Catastrophic collapse |

Rule of thumb: `conc ≤ target_QPS × avg_latency_s` to stay below the connection saturation point.

### Fan-out (cell 9)

4 workers × `initial_concurrency=125` = 500 total connections → ~528 QPS inference-window, 0 errors.
Do not exceed 4 workers at conc=125 — 8 workers causes endpoint congestion (per-worker QPS halves, 8 % rate limits).

### How to use the results

* Use `qps` and `elapsed_s` for end-to-end throughput.
* Use `rate_limited` and `errors` to verify no endpoint pressure.
* Cell 12 queries the output table for a live summary grouped by method and auth mode.

In [0]:
print(f"── recent results: {OUTPUT_TABLE} ──")
display(spark.sql(f"""
SELECT method, auth_mode, {ID_COLUMN}, request_latency_ms, size(matched_account_ids) AS hit_count, processed_at
FROM {OUTPUT_TABLE}
ORDER BY processed_at DESC
LIMIT 10
"""))

print("── aggregated by method and auth ──")
display(spark.sql(f"""
SELECT method, auth_mode, COUNT(*) AS rows_written, ROUND(AVG(request_latency_ms), 1) AS avg_request_ms, MAX(processed_at) AS latest_processed_at
FROM {OUTPUT_TABLE}
GROUP BY method, auth_mode
ORDER BY latest_processed_at DESC, method, auth_mode
"""))